## Week 2 day 3 homework
### AI assistant that generates code in a Ruby on Rails codebase
This is a chatbot can help you integrate the Shopify API to
an existing Ruby on Rails project if you mention the word Shopify.

It provides an example dinamically to the Context so the model
performs better, this is a basic example of multi-shot promping.

It uses Gradio for the Chat UI and
keeps track of the chat history using OpenAI messages format.

The model used is local ollama `llama3.1:8b`.


In [ ]:
# imports
import gradio as gr
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.1:8b"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

SYSTEM_MESSAGE = """
You are a helpful assistant that is an expert in Ruby on Rails codebases.
Give short precise answers and include the code blocks that the user asks for.
If the user doesn't know where to start guide them to the files that need to be created.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def chat(message, history):
    relevant_system_message = SYSTEM_MESSAGE
    if 'shopify' in message.lower():
        relevant_system_message += """
            You provide the Ruby on Rails scaffolding commands to build each database
            model needed to integrate Shopify API to the existing Ruby on Rails codebase.
            You also provide the code blocks for the helper methods that are needed to integrate Shopify API.
        """

    chat_history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": relevant_system_message}] + chat_history + [{"role": "user", "content": message}]
    stream = ollama.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

gr.ChatInterface(
    fn=chat,
    examples=["Which models have to be created to integrate Shopify API in a Ruby on Rails codebase?"],
    title="Ruby on Rails Assistant").launch(inbrowser=True)
